### General Process with steps 

1. read team_stats parquet file from each season folder under the data folder
2. filter by season column, make sure its == 'REG'
3. make copy of dataframe, sort by team and week
4. calculate fields and create new columns in copy of table
5. left or inner join with odds data by season, team, and week
6. output new derived df to same folder destination in parquet file


In [7]:

import os
import pandas as pd

target_directory = 'data/raw'
years = list(range(2016, 2025))

for root, dirs, files in os.walk(target_directory):
    for folder_name in dirs:
        folder_path = os.path.join(root, folder_name)
        print(f"Found folder: {folder_path}")
        # You can perform actions on the folder here

Found folder: data/raw\2015
Found folder: data/raw\2016
Found folder: data/raw\2017
Found folder: data/raw\2018
Found folder: data/raw\2019
Found folder: data/raw\2020
Found folder: data/raw\2021
Found folder: data/raw\2022
Found folder: data/raw\2023
Found folder: data/raw\2024


In [ ]:
def populate_folders (years, base_path='data/raw'):
    
    TEAM_TO_ABBR = {
    'Arizona': 'ARI',
    'Atlanta': 'ATL',
    'Baltimore': 'BAL',
    'Buffalo': 'BUF',
    'Carolina': 'CAR',
    'Chicago': 'CHI',
    'Cincinnati': 'CIN',
    'Cleveland': 'CLE',
    'Dallas': 'DAL',
    'Denver': 'DEN',
    'Detroit': 'DET',
    'Green Bay': 'GB',
    'Houston': 'HOU',
    'Indianapolis': 'IND',
    'Jacksonville': 'JAX',
    'Kansas City': 'KC',
    'Miami': 'MIA',
    'Minnesota': 'MIN',
    'New England': 'NE',
    'New Orleans': 'NO',
    'NY Giants': 'NYG',
    'NY Jets': 'NYJ',
    'Philadelphia': 'PHI',
    'Pittsburgh': 'PIT',
    'Seattle': 'SEA',
    'San Francisco': 'SF',
    'Tampa Bay': 'TB',
    'Tennessee': 'TEN',
    'Washington': 'WAS',
    
    # Historical relocations
    'Oakland': 'LV',
    'St Louis': 'LA',
    'San Diego': 'LAC',
    'LA Chargers': 'LAC',
    'LA Rams': 'LA',
    'Los Angeles': 'LA',  # This should work fine now
    'Las Vegas': 'LV',
}

    
    for year in years:
        season_folder = os.path.join(base_path, str(year))
        os.makedirs(season_folder, exist_ok=True)
        
        odds_data_names_norm = pd.read_parquet('data/raw/odds_names_norm.parquet')
        
        team_stats = pd.read_parquet(f"data/raw/{year}/{'team_stats'}.parquet")
        
        team_stats_reg_szn = team_stats[team_stats['season_type'] == 'REG']
        
        team_stats_filtered = team_stats_reg_szn.sort_values(by=['team', 'week'], ascending=True)
        
        team_stats_copy = team_stats_filtered.copy(deep=True)
        
        total_att_carr = team_stats_copy['attempts'] + team_stats_copy['carries']

        attempts = team_stats_copy['attempts']


        team_stats_copy['pass_rate'] = np.divide(
            attempts,
            total_att_carr,
            out= np.zeros_like(attempts, dtype=float),
            where= total_att_carr != 0
        )


        team_stats_copy['yards_per_att'] = np.divide(
            team_stats_copy['passing_yards'],
            attempts,
            out= np.zeros_like(attempts, dtype=float),
            where=  attempts!= 0)


        team_stats_copy['td_rate'] = np.divide(
            team_stats_copy['passing_tds'],
            attempts,
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )


        team_stats_copy['int_rate'] = np.divide(
            team_stats_copy['passing_interceptions'],
            attempts,
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )

        att_sacks_suff = attempts + team_stats_copy['sacks_suffered']

        team_stats_copy['sack_rate'] = np.divide(
            team_stats_copy['sacks_suffered'],
            att_sacks_suff,
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )

        team_stats_copy['rush_ypc'] = np.divide(
            team_stats_copy['rushing_yards'],
            team_stats_copy['carries'],
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )

        first_downs = team_stats_copy['passing_first_downs'] + team_stats_copy['rushing_first_downs']

        team_stats_copy['explosive_proxy'] = np.divide(
            first_downs,
            total_att_carr,
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )
        team_stats_copy['off_epa'] = team_stats_copy['passing_epa'] + team_stats_copy['rushing_epa'] + team_stats_copy['receiving_epa']

        
        team_stats_copy['epa_per_play'] = np.divide(
            team_stats_copy['off_epa'],
            total_att_carr,
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )
        


        team_stats_copy['adot'] = np.divide(
            team_stats_copy['passing_air_yards'],
            attempts,
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )

        team_stats_copy['yac_per_rec'] =  np.divide(
            team_stats_copy['passing_yards_after_catch'],
            team_stats_copy['receptions'],
            out= np.zeros_like(attempts, dtype=float),
            where= attempts!=0
            )
        
        
        merged_df = team_stats_copy.merge(
                odds_data_names_norm,
                on=['season', 'team', 'week'],
                how='left'
            )




        
        
        
        
        file_name = f'{year}_weekly_offense_and_odds.parquet'
        
        file_path = os.path.join(season_folder, file_name)
        
    return merged_df.to_parquet(file_path)
        
        
        
        

In [ ]:
total_att_carr = team_stats_copy['attempts'] + team_stats_copy['carries']

attempts = team_stats_copy['attempts']


team_stats_copy['pass_rate'] = np.divide(
    attempts,
    total_att_carr,
    out= np.zeros_like(attempts, dtype=float),
    where= total_att_carr != 0
)


team_stats_copy['yards_per_att'] = np.divide(
    team_stats_copy['passing_yards'],
    attempts,
    out= np.zeros_like(attempts, dtype=float),
    where=  attempts!= 0)


team_stats_copy['td_rate'] = np.divide(
    team_stats_copy['passing_tds'],
    attempts,
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )


team_stats_copy['int_rate'] = np.divide(
    team_stats_copy['passing_interceptions'],
    attempts,
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )

att_sacks_suff = attempts + team_stats_copy['sacks_suffered']

team_stats_copy['sack_rate'] = np.divide(
    team_stats_copy['sacks_suffered'],
    att_sacks_suff,
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )

team_stats_copy['rush_ypc'] = np.divide(
    team_stats_copy['rushing_yards'],
    team_stats_copy['carries'],
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )

first_downs = team_stats_copy['passing_first_downs'] + team_stats_copy['rushing_first_downs']

team_stats_copy['explosive_proxy'] = np.divide(
    first_downs,
    total_att_carr,
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )
team_stats_copy['off_epa'] = team_stats_copy['passing_epa'] + team_stats_copy['rushing_epa'] + team_stats_copy['receiving_epa']

 
team_stats_copy['epa_per_play'] = np.divide(
    team_stats_copy['off_epa'],
    total_att_carr,
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )
 


team_stats_copy['adot'] = np.divide(
    team_stats_copy['passing_air_yards'],
    attempts,
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )

team_stats_copy['yac_per_rec'] =  np.divide(
    team_stats_copy['passing_yards_after_catch'],
    team_stats_copy['receptions'],
    out= np.zeros_like(attempts, dtype=float),
    where= attempts!=0
    )



